# Initial ML Solution — Resumable Colab Run

This notebook runs four experiments on the official AVeriTeC train/dev setup: pure BGE dense retrieval, BM25+dense reciprocal-rank fusion (RRF), DistilBERT with BM25 evidence, and DistilBERT with hybrid evidence.

It writes to the output tree named initial_ml_solution_rerun_bs16. Keep RUN_TAG, run names, and model settings unchanged to resume after a Colab interruption. Dense retrieval saves each completed claim and shows claim/passage progress; hybrid fusion shows claim progress; transformer runs show tokenization and optimizer-step progress. The experiment scripts run in this notebook kernel, so errors and progress bars appear directly in their cells.

Use a Colab GPU runtime. Restart the runtime before running this revised notebook, then run the setup cells in order. The model cache uses Colab's local disk, while experiment checkpoints use Drive. Dense embedding batch size is 16 to reduce GPU memory use. Pure dense ranking of the full claim-specific corpus is expensive; each completed claim is resumable. The BM25 evidence verifier is independent of dense retrieval, so you may run its cell immediately after the setup and helper cells to obtain the first trained-model result.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/muyishen2040/10718-ML-In-Practice.git'
REPO_DIR = Path('/content/10718-ML-In-Practice')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import json

DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
PROCESSED = DATA_ROOT / 'processed' / 'averitec'
RAW_ARCHIVES = DATA_ROOT / 'raw' / 'averitec'
RUN_TAG = 'rerun_bs16'  # Keep unchanged to resume these runs.
RUN_ROOT = DATA_ROOT / 'outputs' / f'initial_ml_solution_{RUN_TAG}'
# Keep model downloads on Colab's local disk. Hugging Face cache files can be
# incomplete/corrupted when stored through Google Drive's mounted filesystem.
# Experiment artifacts and resumable checkpoints remain on Drive under RUN_ROOT.
HF_CACHE = Path('/content/huggingface_cache')
HF_CACHE.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['SENTENCE_TRANSFORMERS_HOME'] = str(HF_CACHE / 'sentence-transformers')
# This is a PyTorch workflow; avoid Transformers probing TensorFlow/Flax at startup.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
print('Hugging Face cache:', os.environ['HF_HOME'])
print('TensorFlow/Flax disabled for this run.')
import torch
print('PyTorch CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A PyTorch CUDA GPU is required for these long runs. In Colab choose Runtime > Change runtime type > GPU, then restart and rerun setup.')

DEV_CORPUS = PROCESSED / 'dev_evidence_corpus.jsonl'
DEV_QRELS = PROCESSED / 'dev_qrels.jsonl'
TRAIN_CLAIMS = PROCESSED / 'train.jsonl'
DEV_CLAIMS = PROCESSED / 'dev.jsonl'
TRAIN_BM25 = DATA_ROOT / 'outputs' / 'bm25' / 'train_bm25_top20_v2' / 'train_rankings.jsonl'
DEV_BM25 = DATA_ROOT / 'outputs' / 'bm25' / 'dev_bm25_top20_v2' / 'dev_rankings.jsonl'
TRAIN_ARCHIVES = [
    RAW_ARCHIVES / 'train_0_999.zip',
    RAW_ARCHIVES / 'train_1000_1999.zip',
    RAW_ARCHIVES / 'train_2000_3067.zip',
]
required_inputs = [DEV_CORPUS, DEV_QRELS, TRAIN_CLAIMS, DEV_CLAIMS, TRAIN_BM25, DEV_BM25, *TRAIN_ARCHIVES]
missing_inputs = [str(path) for path in required_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError('Missing expected Drive inputs:\n' + '\n'.join(missing_inputs))
required_scripts = [REPO_DIR / 'scripts' / name for name in ('run_dense_retrieval.py', 'run_hybrid_retrieval.py', 'train_transformer_verifier.py')]
missing_scripts = [str(path) for path in required_scripts if not path.is_file()]
if missing_scripts:
    raise FileNotFoundError('Missing runner scripts in the repository checkout:\n' + '\n'.join(missing_scripts))
print('Input checks passed.')
print('Run output root:', RUN_ROOT)

## Run helpers

Each experiment runs directly in the notebook kernel. The scripts' progress bars and full Python tracebacks appear in the cell. After interruption, reconnect, rerun setup, and rerun the interrupted experiment cell with the same run root/name/settings. The dense script checkpoints after every completed claim; the transformer resumes from its last complete Trainer checkpoint.

In [ ]:
import gc
import runpy
import sys
import time

def run_command(label, command):
    # Accept the existing command lists, but execute their scripts in this kernel.
    parts = [str(part) for part in command]
    if parts[:2] != ['python', '-u'] or len(parts) < 3:
        raise ValueError(f'Expected a Python runner command, got: {parts}')
    script = (REPO_DIR / parts[2]).resolve()
    if not script.is_relative_to(REPO_DIR.resolve()) or not script.is_file():
        raise FileNotFoundError(f'Runner script missing or outside repository: {script}')
    previous_argv = sys.argv[:]
    sys.argv = [str(script), *parts[3:]]
    print(f'\n=== {label} ===', flush=True)
    print('Runner:', script, flush=True)
    print('Arguments:', ' '.join(parts[3:]), flush=True)
    started = time.monotonic()
    try:
        runpy.run_path(str(script), run_name='__main__')
    except KeyboardInterrupt:
        print(f'{label} interrupted. Rerun this cell unchanged to resume.', flush=True)
        raise
    finally:
        sys.argv = previous_argv
        gc.collect()
        torch.cuda.empty_cache()
    print(f'{label} completed in {(time.monotonic() - started) / 60:.1f} minutes.', flush=True)

def outputs_complete(paths):
    for path in map(Path, paths):
        if not path.is_file() or path.stat().st_size == 0:
            return False
        if path.suffix == '.json':
            try:
                json.loads(path.read_text(encoding='utf-8'))
            except (OSError, ValueError):
                return False
    return True

def run_unless_complete(label, command, outputs):
    if outputs_complete(outputs):
        print(f'{label}: completed artifacts already exist; reusing them.')
        return
    run_command(label, command)
    if not outputs_complete(outputs):
        raise RuntimeError(f'{label} returned without complete artifacts: {outputs}')

## Experiment 1 — Pure dense retrieval on development

Ranks passages within each claim's candidate pool, retains the top 20, and evaluates URL-proxy hit@3/MRR. A new run name/output tree keeps this rerun separate from the earlier interrupted attempt. A future interruption in this run resumes from its own partial checkpoint.

In [ ]:
DENSE_MODEL = 'BAAI/bge-small-en-v1.5'
DENSE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
DENSE_BATCH_SIZE = 16
DENSE_MAX_SEQ_LENGTH = 384

DENSE_DEV_DIR = RUN_ROOT / 'dense' / 'dev_bge_small_v15'
dense_dev_rankings = DENSE_DEV_DIR / 'dev_rankings.jsonl'
dense_dev_metrics = DENSE_DEV_DIR / 'dev_metrics.json'
dense_dev_manifest = DENSE_DEV_DIR / 'dev_manifest.json'
dense_dev_command = [
    'python', '-u', 'scripts/run_dense_retrieval.py',
    '--split', 'dev', '--data-root', DATA_ROOT, '--corpus', DEV_CORPUS, '--qrels', DEV_QRELS,
    '--output-dir', RUN_ROOT / 'dense', '--run-name', 'dev_bge_small_v15',
    '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
    '--device', 'cuda',
    '--ranking-k', '20', '--metric-k', '3', '--batch-size', str(DENSE_BATCH_SIZE),
    '--max-seq-length', str(DENSE_MAX_SEQ_LENGTH), '--checkpoint-every', '1', '--resume',
]
run_unless_complete(
    'Dense retrieval / dev',
    dense_dev_command,
    [dense_dev_rankings, dense_dev_metrics, dense_dev_manifest],
)

## Experiment 2 — BM25 + dense reciprocal-rank fusion

Fuses saved BM25 and dense top-20 rankings using fixed RRF constant 60. This deterministic step is quick and safe to recompute if interrupted.

In [ ]:
HYBRID_DEV_DIR = RUN_ROOT / 'hybrid' / 'dev_rrf60'
hybrid_dev_rankings = HYBRID_DEV_DIR / 'hybrid_rankings.jsonl'
hybrid_dev_metrics = HYBRID_DEV_DIR / 'hybrid_metrics.json'
hybrid_dev_manifest = HYBRID_DEV_DIR / 'hybrid_manifest.json'
hybrid_dev_command = [
    'python', '-u', 'scripts/run_hybrid_retrieval.py',
    '--bm25-rankings', DEV_BM25, '--dense-rankings', dense_dev_rankings, '--qrels', DEV_QRELS,
    '--output-dir', RUN_ROOT / 'hybrid', '--run-name', 'dev_rrf60',
    '--rrf-constant', '60', '--ranking-k', '20', '--metric-k', '3',
]
run_unless_complete(
    'Hybrid retrieval / dev',
    hybrid_dev_command,
    [hybrid_dev_rankings, hybrid_dev_metrics, hybrid_dev_manifest],
)

## Experiment 3 — DistilBERT verifier with BM25 top-three evidence

Tunes two learning rates × two epoch settings on a stratified 10% holdout from official train only. It selects by Macro-F1 (false-reassurance/false-alarm rates break ties), refits on all usable train claims, then evaluates on official dev. Tokenization and optimizer-step progress are displayed; checkpoints are stored under Drive.

In [ ]:
TRANSFORMER_DIR = RUN_ROOT / 'transformer'
transformer_bm25_dir = TRANSFORMER_DIR / 'bm25_top3'
transformer_bm25_metrics = transformer_bm25_dir / 'final_dev_metrics.json'
transformer_bm25_predictions = transformer_bm25_dir / 'final_dev_predictions.jsonl'
transformer_bm25_manifest = transformer_bm25_dir / 'transformer_manifest.json'
transformer_bm25_command = [
    'python', '-u', 'scripts/train_transformer_verifier.py',
    '--data-root', DATA_ROOT, '--train-rankings', TRAIN_BM25, '--eval-rankings', DEV_BM25,
    '--output-dir', TRANSFORMER_DIR, '--run-name', 'bm25_top3',
    '--model', 'distilbert-base-uncased', '--max-length', '512',
    '--claim-token-budget', '96', '--evidence-token-budget', '110',
    '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
    '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100',
    '--seed', '718', '--resume',
]
run_unless_complete(
    'Transformer verifier / BM25 evidence',
    transformer_bm25_command,
    [transformer_bm25_metrics, transformer_bm25_predictions, transformer_bm25_manifest],
)

## Experiment 4a — Dense train rankings

Processes all three official train archive shards in one resumable invocation. This is likely the longest run. If interrupted, rerun this cell unchanged; completed claims are skipped. Then run the finalization cell below.

In [ ]:
DENSE_TRAIN_DIR = RUN_ROOT / 'dense' / 'train_bge_small_v15'
dense_train_rankings = DENSE_TRAIN_DIR / 'train_rankings.jsonl'
dense_train_metrics = DENSE_TRAIN_DIR / 'train_metrics.json'
dense_train_manifest = DENSE_TRAIN_DIR / 'train_manifest.json'
dense_train_command = [
    'python', '-u', 'scripts/run_dense_retrieval.py',
    '--split', 'train', '--data-root', DATA_ROOT,
]
for archive_path in TRAIN_ARCHIVES:
    dense_train_command.extend(['--archive', archive_path])
dense_train_command.extend([
    '--output-dir', RUN_ROOT / 'dense', '--run-name', 'train_bge_small_v15',
    '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
    '--device', 'cuda',
    '--ranking-k', '20', '--metric-k', '3', '--batch-size', str(DENSE_BATCH_SIZE),
    '--max-seq-length', str(DENSE_MAX_SEQ_LENGTH), '--checkpoint-every', '1',
    '--max-words', '160', '--overlap-words', '40', '--resume',
])
run_command('Dense retrieval / train archives', dense_train_command)
print('Archive pass finished. Run the following finalization cell before using train rankings.')

In [ ]:
if not outputs_complete([dense_train_rankings, dense_train_metrics, dense_train_manifest]):
    run_command('Finalize dense train rankings', [
        'python', '-u', 'scripts/run_dense_retrieval.py',
        '--split', 'train', '--data-root', DATA_ROOT,
        '--output-dir', RUN_ROOT / 'dense', '--run-name', 'train_bge_small_v15',
        '--model', DENSE_MODEL, '--query-prefix', DENSE_QUERY_PREFIX,
        '--ranking-k', '20', '--metric-k', '3', '--batch-size', str(DENSE_BATCH_SIZE),
        '--max-seq-length', str(DENSE_MAX_SEQ_LENGTH), '--max-words', '160', '--overlap-words', '40',
        '--resume', '--finalize',
    ])
else:
    print('Dense train rankings already finalized.')

## Experiment 4b — Train fusion and hybrid-evidence verifier

Fuse train BM25+dense rankings, then train the same classifier on hybrid top-three evidence. Fusion is deterministic; transformer tuning/refit resumes from its Drive checkpoints.

In [ ]:
HYBRID_TRAIN_DIR = RUN_ROOT / 'hybrid' / 'train_rrf60'
hybrid_train_rankings = HYBRID_TRAIN_DIR / 'hybrid_rankings.jsonl'
hybrid_train_metrics = HYBRID_TRAIN_DIR / 'hybrid_metrics.json'
hybrid_train_manifest = HYBRID_TRAIN_DIR / 'hybrid_manifest.json'
hybrid_train_command = [
    'python', '-u', 'scripts/run_hybrid_retrieval.py',
    '--bm25-rankings', TRAIN_BM25, '--dense-rankings', dense_train_rankings,
    '--output-dir', RUN_ROOT / 'hybrid', '--run-name', 'train_rrf60',
    '--rrf-constant', '60', '--ranking-k', '20', '--metric-k', '3',
]
run_unless_complete(
    'Hybrid retrieval / train',
    hybrid_train_command,
    [hybrid_train_rankings, hybrid_train_metrics, hybrid_train_manifest],
)

In [ ]:
transformer_hybrid_dir = TRANSFORMER_DIR / 'hybrid_top3'
transformer_hybrid_metrics = transformer_hybrid_dir / 'final_dev_metrics.json'
transformer_hybrid_predictions = transformer_hybrid_dir / 'final_dev_predictions.jsonl'
transformer_hybrid_manifest = transformer_hybrid_dir / 'transformer_manifest.json'
transformer_hybrid_command = [
    'python', '-u', 'scripts/train_transformer_verifier.py',
    '--data-root', DATA_ROOT, '--train-rankings', hybrid_train_rankings, '--eval-rankings', hybrid_dev_rankings,
    '--output-dir', TRANSFORMER_DIR, '--run-name', 'hybrid_top3',
    '--model', 'distilbert-base-uncased', '--max-length', '512',
    '--claim-token-budget', '96', '--evidence-token-budget', '110',
    '--validation-fraction', '0.10', '--learning-rates', '2e-5,5e-5', '--epoch-grid', '2,3',
    '--batch-size', '8', '--gradient-accumulation-steps', '2', '--checkpoint-steps', '100',
    '--seed', '718', '--resume',
]
run_unless_complete(
    'Transformer verifier / hybrid evidence',
    transformer_hybrid_command,
    [transformer_hybrid_metrics, transformer_hybrid_predictions, transformer_hybrid_manifest],
)

## Compare completed results

Uses the same official dev split. The official test split remains untouched.

In [ ]:
result_paths = {
    'BM25 retrieval': DATA_ROOT / 'outputs' / 'bm25' / 'dev_bm25_top20_v2' / 'dev_metrics.json',
    'Dense retrieval': dense_dev_metrics,
    'Hybrid retrieval': hybrid_dev_metrics,
    'Qwen + BM25': DATA_ROOT / 'outputs' / 'local_llm' / 'dev_qwen3_4b_bm25_top3_v2' / 'classify_top3_metrics.json',
    'TF-IDF LR + BM25': DATA_ROOT / 'outputs' / 'tfidf_logreg' / 'retrieved_train_to_dev_v2' / 'retrieved_train_to_dev_metrics.json',
    'Transformer + BM25': transformer_bm25_metrics,
    'Transformer + hybrid': transformer_hybrid_metrics,
}
for name, path in result_paths.items():
    path = Path(path)
    if not path.is_file():
        print(f'{name}: not run yet')
        continue
    report = json.loads(path.read_text(encoding='utf-8'))
    errors = report.get('decision_error_rates') or {}
    print(name, {
        'URL_proxy_hit@3': report.get('evidence_hit_at_3'),
        'MRR': report.get('mrr'),
        'Macro-F1': report.get('macro_f1'),
        'accuracy': report.get('accuracy'),
        'false_reassurance': errors.get('false_reassurance_rate'),
        'false_alarm': errors.get('false_alarm_rate'),
    })